## Task 1 – Pregunta 1.1

### 1. ¿Qué es Data Augmentation y por qué no son datos falsos?

El **Data Augmentation** es una técnica que consiste en generar nuevas versiones de las imágenes existentes aplicando pequeñas transformaciones (como rotaciones leves, cambios de brillo o recortes), sin alterar el contenido clínico relevante.(s.f, 2024)

**Analogía para el médico:**

Un paciente al que se observa en diferentes condiciones:
- con distinta iluminación,
- en una posición ligeramente diferente,
- o con variaciones normales del equipo de rayos X.

El paciente sigue siendo el mismo, y su diagnóstico no cambia.  
Lo único que cambia es **cómo se ve la imagen**, no **lo que contiene clínicamente**.

Por eso, **no estamos inventando datos falsos**, sino mostrando al modelo las mismas radiografías en condiciones ligeramente distintas, tal como ocurriría en la práctica real en clínicas rurales.

Esto ayuda a que el modelo no “memorice” una imagen específica, sino que aprenda **patrones generales de enfermedad**.

---

### 2. Transformaciones válidas en radiografías de tórax

A continuación, tres transformaciones apropiadas para este dominio:

#### a) Rotaciones leves ($$\pm 5^grados$$ a $$\pm 10^grados$$)
- **Justificación:** En la práctica, los pacientes no siempre están perfectamente alineados.
- **Impacto positivo:** El modelo aprende a reconocer anomalías aunque la imagen esté ligeramente inclinada.

---

#### b) Ajustes de brillo y contraste
- **Justificación:** Diferentes equipos de rayos X o condiciones de exposición generan imágenes más claras u oscuras.
- **Impacto positivo:** Mejora la robustez del modelo ante variabilidad entre clínicas rurales.

---

#### c) Traslaciones o recortes leves (shifts)
- **Justificación:** El encuadre del tórax puede variar dependiendo del técnico.
- **Impacto positivo:** El modelo no depende de que los pulmones estén exactamente centrados.

---

### Transformación NO recomendada

#### d) Volteo horizontal (horizontal flip)
- **Problema:** Invierte izquierda y derecha del cuerpo.
- **Riesgo clínico:** Puede alterar información anatómica crítica (por ejemplo, ubicación de lesiones o del corazón).
- **Consecuencia:** El modelo podría aprender patrones incorrectos y comprometer la interpretación médica.

---

### 3. ¿Es suficiente el Data Augmentation?

No, **el Data Augmentation por sí solo no garantiza una buena generalización**.

Es una herramienta que puede ayudar bastante, pero debe complementarse:

#### a) Calidad y representatividad del dataset
- 800 imágenes pueden ser insuficientes si no cubren **diversidad** (edades, equipos, condiciones clínicas).
- “Garbage in, garbage out”: el modelo solo puede aprender lo que ve.

---

#### b) Uso de Transfer Learning
- Utilizar modelos preentrenados (por ejemplo, en ImageNet o datasets médicos).
- Permite aprovechar conocimiento previo y reducir el riesgo de sobreajuste.

---

#### c) Regularización
Ejemplos:
- Dropout  
- Weight decay  

Estas técnicas ayudan a evitar que el modelo memorice:

$$
L = L_{datos} + \lambda ||w||^2
$$

---

#### d) Validación adecuada
- Separar correctamente entrenamiento, validación y prueba.
- Idealmente incluir datos de diferentes clínicas.

---

#### e) Arquitectura del modelo
- Dado que el sistema debe correr **sin GPU**, se requieren modelos livianos (por ejemplo, MobileNet, como transfer o primer modelo).
- Un modelo muy grande se puede sobreajustar con pocos datos.


## Task 1 – Pregunta 1.2

### 4. ¿Cuándo comienza el sobreajuste (overfitting)?

El sobreajuste comienza aproximadamente **después de la época 10 (alrededor de la 15)**.

- Hasta la época 10: ambas pérdidas **disminuyen**, lo cual indica buen aprendizaje.
- A partir de la época 15 porque la **loss de validación empieza a subir** (0.38 → 0.89)

Esto indica que el modelo está **memorizando el dataset de entrenamiento** pero perdiendo capacidad de generalizar a nuevos datos.

### 5. Estrategias de regularización

#### a) Dropout

**Idea:** Durante el entrenamiento, se "apagan" aleatoriamente neuronas con cierta probabilidad.

**Intuición matemática:**
- Evita que el modelo dependa demasiado de combinaciones específicas de neuronas.
- Reduce la co-adaptación de características.

En términos simples, el modelo deja de aprender patrones demasiado específicos.

**Impacto esperado en las curvas:**
- La loss de entrenamiento subiría ligeramente (entrenamiento más difícil).
- La loss de validación disminuiría o se estabilizaría.
- Se reduciría la brecha entre ambas curvas.


#### b) Regularización L2 (Weight Decay)

Penaliza pesos grandes en el modelo.

$$
L = L_{datos} + \lambda ||w||^2
$$

- Obliga al modelo a mantener pesos pequeños.
- Evita funciones demasiado complejas o altamente ajustadas al ruido.

Por lo tanto:
- La loss de entrenamiento no bajaría tan agresivamente.
- La loss de validación dejaría de crecer tan rápido.
- Curvas más suaves y cercanas entre sí.


### 6. Riesgo médico del sobreajuste

Desde una perspectiva médica, este patrón es **especialmente peligroso** por varias razones:

#### a) Falsa sensación de precisión
El modelo parece muy bueno en entrenamiento (loss muy baja), pero en realidad:
- Puede fallar en pacientes reales.
- No generaliza a nuevas clínicas o equipos.


#### b) Diagnósticos incorrectos
- **Falsos negativos:** no detectar neumonía → paciente sin tratamiento.
- **Falsos positivos:** diagnosticar enfermedad inexistente → tratamientos innecesarios.

En ambos casos hay consecuencias sumamente graves.


#### c) Variabilidad en clínicas rurales
El modelo se usará en entornos con:
- diferentes máquinas de rayos X,
- distintas condiciones de adquisición


#### d) Impacto en la confianza del sistema
Si los médicos notan errores inconsistentes:
- perderán confianza en la herramienta,
- lo cual compromete la adopción del sistema MediScan.


## Task 1 – Pregunta 1.3

### 7. Accuracy de un modelo naive

El dataset de prueba contiene:
- 700 imágenes **Normales**
- 150 imágenes con **Neumonía**
- Total: 850 imágenes

Un modelo naive que siempre predice **"Normal"**:
- Acierta las 700 imágenes normales
- Falla las 150 con neumonía

$$
Accuracy = \frac{700}{850} \approx 0.8235 \approx 82.35\%
$$

- Un modelo extremadamente simple logra **82.35% de accuracy** sin detectar ningún caso de neumonía.
- El 94% reportado es solo ~12% mejor que este baseline trivial.

Esto revela que el **accuracy puede ser engañoso** en datasets desbalanceados.

---

### 8. Métricas más informativas en contexto médico

En problemas médicos, el objetivo no es solo atinarle, sino **detectar correctamente y constantemente la enfermedad**.

#### a) Sensibilidad (Recall para neumonía)

$$
Recall = \frac{TP}{TP + FN}
$$

- Mide qué proporción de pacientes con neumonía fue correctamente detectada.
- **Relevancia clínica:**  
  Un bajo recall implica muchos **falsos negativos**, es decir, pacientes enfermos no diagnosticados.

En medicina, esto puede significar **no tratar a un paciente que lo necesita e incluso exponerlos a sugestionses**.

---

#### b) F1-Score

$$
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
$$

- Balancea precisión (evitar falsos positivos) y recall (evitar falsos negativos).
- **Relevancia clínica:**
  - Penaliza modelos que solo optimizan una métrica.
  - Refleja mejor el desempeño real en escenarios desbalanceados.

---

### ¿Por qué son mejores que accuracy?

- El **accuracy está dominado por la clase mayoritaria** (normales).
- Un modelo puede tener alta accuracy y aún así:
  - fallar en detectar neumonía,
  - ser clínicamente inútil.

En este contexto, **detectar correctamente la enfermedad es más importante que acertar casos normales**.+

**Al final puede ser un clásico ejemplo de accuracy vs precision, pero ahora es mas delicado por el área de evaluación.**

---

### 9. Respuesta al inversionista

Un 94% de accuracy es un buen indicador inicial, pero no es suficiente por sí solo para evaluar la confiabilidad del modelo en un contexto médico. Nuestro dataset está desbalanceado, lo que significa que un modelo puede obtener alta precisión sin necesariamente detectar bien los casos de neumonía, que son los más críticos. Por eso, también evaluamos métricas como la sensibilidad y el F1-score, que reflejan mejor la capacidad del sistema para identificar pacientes enfermos. Nuestro enfoque es asegurar no solo un buen desempeño global, sino también minimizar errores clínicamente riesgosos antes de cualquier despliegue.